# Module C — Retrieval Models

1. Model 1: Lexical Retrieval (BM25)
2. Model 2: Fuzzy/Transliteration Matching
3. Model 3: Semantic Matching
4. Model 4: Hybrid Ranking

In [1]:
import feedparser
import json
from tqdm import tqdm
import os
from bs4 import BeautifulSoup
import html
import re
import pickle
from rank_bm25 import BM25Okapi

## 3. Configuration

### 3.1 Path

In [2]:
EN_PATH = r"E:\DM\Cross-Lingual-Information-Retrieval-System\data\document_en_clean.json"
BN_PATH = r"E:\DM\Cross-Lingual-Information-Retrieval-System\data\document_bn_clean.json"

EN_INDEX_OUT='bm25_en.pkl'
BN_INDEX_OUT='bm25_bn.pkl'

### 3.2 Tokenizer

#### 3.2.1 English Tokenizer

In [3]:
def tokenize_en(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

#### 3.2.1 Bangla Tokenizer

In [4]:
def tokenize_bn(text):
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

### 3.3 Build and Save English BM25 index

In [5]:
with open(EN_PATH, 'r', encoding='utf-8') as f:
    docs_en = json.load(f)

corpus_end=[]
doc_ids_en=[]

for doc in docs_en:
    title = doc.get("title", "")
    body = doc.get("body", "")

    full_text = (title + " " + body).strip()

    corpus_end.append(tokenize_en(full_text))
    doc_ids_en.append(doc.get("doc_id"," "))

bm25_en = BM25Okapi(corpus_end)

with open(EN_INDEX_OUT, 'wb') as f:
    pickle.dump({"bm25": bm25_en, "doc_ids": doc_ids_en, "docs": docs_en}, f)

print(f"English docs indexing:" ,len(doc_ids_en))
print(f"BM25 English index saved to:" ,EN_INDEX_OUT)

English docs indexing: 153
BM25 English index saved to: bm25_en.pkl


### 3.4 Build and Save Bangla BM25 index

In [6]:
with open(BN_PATH, 'r', encoding='utf-8') as f:
    docs_bn = json.load(f)

corpus_end=[]
doc_ids_bn=[]

for doc in docs_bn:
    title = doc.get("title", "")
    body = doc.get("body", "")

    full_text = (title + " " + body).strip()

    corpus_end.append(tokenize_bn(full_text))
    doc_ids_bn.append(doc.get("doc_id"," "))

bm25_bn = BM25Okapi(corpus_end)

with open(BN_INDEX_OUT, 'wb') as f:
    pickle.dump({"bm25": bm25_bn, "doc_ids": doc_ids_bn, "docs": docs_bn}, f)

print(f"Bengali docs indexing:" ,len(doc_ids_bn))
print(f"BM25 Bengali index saved to:" ,BN_INDEX_OUT)

Bengali docs indexing: 164
BM25 Bengali index saved to: bm25_bn.pkl


### 3.5 Load Indexes and Search Function

#### 3.5.1 Load Indexes

In [7]:
def load_index(path):
    with open(path, "rb") as f:
        return pickle.load(f)

en_pack = load_index(EN_INDEX_OUT)
bn_pack = load_index(BN_INDEX_OUT)

bm25_en = en_pack["bm25"]
doc_ids_en = en_pack["doc_ids"]
docs_en = en_pack["docs"]

bm25_bn = bn_pack["bm25"]
doc_ids_bn = bn_pack["doc_ids"]
docs_bn = bn_pack["docs"]

#### 3.5.2 English Search Function

In [8]:
def search_en(query, top_k=5):
    q = tokenize_en(query)
    scores = bm25_en.get_scores(q)
    idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{"doc_id": doc_ids_en[i], "score": float(scores[i]), "title": docs_en[i].get("title",""), "url": docs_en[i].get("url","")} for i in idx]


#### 3.5.3 Bangla Search Function

In [9]:
def search_bn(query, top_k=5):
    q = tokenize_bn(query)
    scores = bm25_bn.get_scores(q)
    idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{"doc_id": doc_ids_bn[i], "score": float(scores[i]), "title": docs_bn[i].get("title",""), "url": docs_bn[i].get("url","")} for i in idx]

### 3.6 Test

#### 3.6.1 Test English Query

In [10]:
for r in search_en("Bangladesh cricket", top_k=5):
    print(r["score"], "|", r["doc_id"], "|", r["title"])
    print("  ", r["url"])

5.933856132560005 | en_000189 | a bangladesh cricket board (bcb) official on thursday said that the board has taken over the chattogram royals franchise following a withdrawal letter submitted by the franchise owner, just a day before the start of the 12th edition of the bangladesh premier league (bpl).
   https://www.thedailystar.net/sports/sports-special/bpl-2026/news/bcb-takes-over-chattogram-franchise-after-owners-withdrawal-letter-4065916
5.733759090234742 | en_000177 | as the tournament gets underway at the sylhet international cricket stadium on friday, controversies continue to surface despite repeated assurances of a smoothly run event.
   https://www.thedailystar.net/sports/sports-special/bpl-2026/news/bpl-12-begins-today-chaos-tow-4066291
5.66419036332829 | en_000182 | noakhali express head coach khaled mahmud and assistant coach talha jubair returned to training on thursday after briefly walking out midway through the session over a shortage of facilities, with the issue la

#### 3.6.2 Test Bangla Query

In [11]:
for r in search_bn("বাংলাদেশ ক্রিকেট", top_k=5):
    print(r["score"], "|", r["doc_id"], "|", r["title"])
    print("  ", r["url"])

5.7852337630911475 | bn_000162 | বিপিএল শুরুর আগমুহূর্তে গুরুত্বপূর্ণ দায়িত্বে নান্নু
   https://www.jagonews24.com/sports/news/1079007
5.498008033997645 | bn_000076 | বক্সিং ডে টেস্টের নেতৃত্বে স্মিথ, নেই কামিন্স-লায়ন
   https://www.risingbd.com/sports/news/633407
5.3337631155419105 | bn_000012 | টিভিতে আজকের খেলা
   https://www.risingbd.com/sports/news/633471
2.9149818534770966 | bn_000129 | ঢাকায় পৌঁছেছে শহীদ শরিফ ওসমান হাদির মরদেহ
   https://bangladeshdiplomat.com/11050/latest/%e0%a6%a2%e0%a6%be%e0%a6%95%e0%a6%be%e0%a6%af%e0%a6%bc-%e0%a6%aa%e0%a7%8c%e0%a6%81%e0%a6%9b%e0%a7%87%e0%a6%9b%e0%a7%87-%e0%a6%b6%e0%a6%b9%e0%a7%80%e0%a6%a6-%e0%a6%b6%e0%a6%b0%e0%a6%bf%e0%a6%ab/
2.8633953491319146 | bn_000062 | নিরাপদ বাংলাদেশ গড়তে চাই: তারেক রহমান
   https://www.risingbd.com/politics/news/633421
